# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [4]:
print(locations_df.shape)
print(locations_df.isna().sum())
locations_df[locations_df['tract_geoid'].isna()]

(16790, 14)
agtno                      0
busnm                      0
busaddr                    0
buscity                    0
buszip                     0
chain                      0
bustype                    0
longitude                  0
latitude                   0
geometry                   0
bg_geoid                   0
tract_geoid                0
region                     0
competitors_within_1_mi    0
dtype: int64


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,region,competitors_within_1_mi


In [5]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [6]:
print(sales_df.shape)
print(sales_df.isna().sum())
sales_df.sample(5)

(14351, 15)
agtno                     0
total_sales               0
first_date                0
last_date                 0
first_pos_date            0
last_pos_date             0
pos_sales_days            0
num_cal_days              0
num_active_days           0
days_before_first_sale    0
days_after_last_sale      0
avg_cal_daily_sales       0
avg_active_daily_sales    0
use_active_period         0
avg_daily_sales           0
dtype: int64


,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
13806,95487,318907.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,873.717808,873.717808,False,873.717808
9111,44141,536735.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,362.0,365,365.0,0.0,0.0,1470.508219,1470.508219,False,1470.508219
4600,112540,1045194.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,2863.545205,2863.545205,False,2863.545205
11093,71677,1004072.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,2750.882192,2750.882192,False,2750.882192
11848,75645,139976.5,2025-01-01,2025-12-16,2025-01-01,2025-06-03,134.0,350,154.0,0.0,196.0,399.932857,908.938312,True,908.938312


## Overture Maps Data

In [7]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df['highest_hierarchy'] = interim_overture_df['taxonomy_hierarchy'].str[0]
print(interim_overture_df['highest_hierarchy'].unique())
print(interim_overture_df['basic_category'].nunique())
interim_overture_df.sample(20)

['services_and_business' 'sports_and_recreation' 'education' 'health_care'
 'arts_and_entertainment' 'shopping' 'lifestyle_services'
 'travel_and_transportation' 'community_and_government'
 'cultural_and_historic' 'geographic_entities' 'food_and_drink' 'lodging'
 None]
201


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen,highest_hierarchy
352,c7b7634d-a2ff-4093-a5b2-c518783c6ee8,b'\x01\x01\x00\x00\x00\xb6\x84|\xd0\xb3/R\xc0\x11p\x08UjvD@',-72.745350,40.925120,hardware_home_and_garden_store,building_supply_store,"[shopping, specialty_store, hardware_home_and_garden_store, building_supply_store]",0.991375,NaN,2025-06-25.0,shopping
2169,934811d9-6aa7-498a-aa03-58508e2faa94,b'\x01\x01\x00\x00\x00\xd7w4\xc4\xe7@R\xc0\xa5\xf8\xf8\x84\xecwD@',-73.014146,40.936906,bank_or_credit_union,bank_or_credit_union,"[services_and_business, financial_service, bank_or_credit_union]",0.919912,NaN,2025-06-25.0,services_and_business
3055,15b4e3ec-ef4e-43aa-8448-917d737b6cb0,b'\x01\x01\x00\x00\x00\x93\xa0\t^\xbe*R\xc0\xab\x03\xc54xwD@',-72.667869,40.933356,dental_clinic,dental_clinic,"[health_care, outpatient_care_facility, dental_clinic]",0.990219,NaN,2025-06-25.0,health_care
3885,ba860c1f-344c-4868-aa64-1dfb7ef3bd45,b'\x01\x01\x00\x00\x00\xae\x91\xd1\xbflwR\xc0t\x93Z (\x1cE@',-73.866013,42.219975,NaN,NaN,None,0.919912,NaN,2025-06-25.0,None
2709,341e7687-5aea-43c6-9a62-76eab3d0279d,b'\x01\x01\x00\x00\x00^-wfB9R\xc0+w\x13:7yD@',-72.894678,40.946998,food_and_beverage_store,seafood_market,"[shopping, food_and_beverage_store, seafood_market]",0.930156,NaN,2025-06-25.0,shopping
3123,0cc7a5e0-ba91-46c2-9e53-0ef3970af350,b'\x01\x01\x00\x00\x00\xbak\t\xf9\xa0)R\xc0\xb9\xfc\x87\xf4\xdbwD@',-72.650450,40.936400,dental_clinic,dental_clinic,"[health_care, outpatient_care_facility, dental_clinic]",0.921292,NaN,2025-06-25.0,health_care
4706,9110f82b-4ec3-4c77-9f8d-3611a69fc355,b'\x01\x01\x00\x00\x00m\x9c\xa0\x8f*wR\xc0\xc7|\x06\xf5)NE@',-73.861973,42.610655,home_service,tree_service,"[services_and_business, home_service, landscaping, tree_service]",0.919912,NaN,2025-06-25.0,services_and_business
3687,edf157bb-6d48-4c50-818e-134755395282,b'\x01\x01\x00\x00\x00\xe1\xdb\xcb\xe8\xcdwR\xc0\xd0R\x89\x1dS\x15E@',-73.871943,42.166599,home_service,contractor,"[services_and_business, home_service, contractor]",0.919912,NaN,2025-06-25.0,services_and_business
2814,fdbea091-35c3-4b48-8026-521747378b0d,b'\x01\x01\x00\x00\x00x\xe1Z/\xfe5R\xc0\x89W\x82Z\xc6xD@',-72.843639,40.943553,real_estate_service,real_estate_agent,"[services_and_business, real_estate_service, real_estate_agent]",0.919912,NaN,2025-06-25.0,services_and_business
1445,1a3a1627-f51b-4797-9bb2-17920480b28e,b'\x01\x01\x00\x00\x00w\xc1;T.)R\xc0\x0f?\x83v\x9cgD@',-72.643453,40.809462,casual_eatery,ice_cream_shop,"[food_and_drink, casual_eatery, dessert_shop, ice_cream_shop]",0.919912,NaN,2025-06-25.0,food_and_drink


In [8]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [9]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,commercial_places_within_1_mi,community_places_within_1_mi
42,78975,BULLVILLE FOOD MART,2887 ROUTE 17K,BULLVILLE,10915,0,CONV. STORE,-74.358904,41.541798,"b'\x01\x01\x00\x00\x00\x1d\xcb""G\xf8\x96R\xc0wI\xdf\xa5Y\xc5D@'",360710109023,36071010902,7,2
70,114489,LORRIES DELI GROCERY CORP,365 E 193RD ST,BRONX,10458,0,DELI-GROCERY,-73.890900,40.863110,b'\x01\x01\x00\x00\x00i\x00o\x81\x04yR\xc0\xdflscznD@',360050399013,36005039901,869,254
21,112587,BOLLA 1193 MYRTLE AVE,1193 MYRTLE AVE,BROOKLYN,11221,282,CONV. STORE,-73.932830,40.697250,b'\x01\x01\x00\x00\x00\x92\xae\x99|\xb3{R\xc0h\x91\xed|?YD@',360470389003,36047038900,1221,353
1,68523,YAHIR & SISTERS GROCERY CORP,165 MCLEAN AVE,YONKERS,10705,0,GROCERY STORE,-73.889960,40.915460,b'\x01\x01\x00\x00\x00\xe2\xaf\xc9\x1a\xf5xR\xc0\xe9e\x14\xcb-uD@',361190013013,36119001301,267,84
78,79240,LA ROSA WINES & LIQUORS,3906 BROADWAY,NEW YORK,10032,0,LIQUOR,-73.941900,40.837960,b'\x01\x01\x00\x00\x00\x8e\x06\xf0\x16H|R\xc0\x97\xad\xf5EBkD@',360610245001,36061024500,759,243


## LEHD Data

In [10]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [11]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,rac_ce01_prop,rac_ce02_prop
1,68523,YAHIR & SISTERS GROCERY CORP,165 MCLEAN AVE,YONKERS,10705,independent,GROCERY STORE,-73.88996,40.91546,b'\x01\x01\x00\x00\x00\xe2\xaf\xc9\x1a\xf5xR\xc0\xe9e\x14\xcb-uD@',361190013013,36119001301,44,667.0,75.0,167.0,425.0,0.112444,0.250375
57,108836,SHUNFA DISCOUNT INC,5309 CHURCH AVE,BROOKLYN,11203,independent,GROCERY STORE,-73.92711,40.65235,b'\x01\x01\x00\x00\x00\xdar.\xc5U{R\xc0\xd6\xc5m4\x80SD@',360470866003,36047086600,123,419.0,49.0,127.0,243.0,0.116945,0.303103


In [12]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [13]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01,wac_ce02,wac_ce03,wac_ce01_prop,wac_ce02_prop
85,114079,NEWTOWN CONVENIENT STORE INC,2931 NEWTOWN AVE,ASTORIA,11102,independent,VARIETY,-73.92141,40.76855,b'\x01\x01\x00\x00\x00E\r\xa6a\xf8zR\xc0\x9f\xab\xad\xd8_bD@',360810069001,36081006900,132,0.0,0.0,0.0,2.0,0.0,0.0,5.0,2.0,0.0,0.0,5.0,5.0,0.0,0.0,3.0,59.0,0.0,92.0,1.0,0.0,174.0,42.0,47.0,85.0,0.241379,0.270115
18,105900,7-ELEVEN #40416,4949 TRANSIT RD,DEPEW,14043,chain,CONV. STORE,-78.69641,42.88078,"b""\x01\x01\x00\x00\x00\xdf\xa6?\xfb\x91\xacS\xc0F|'f\xbdpE@""",360290142071,36029014207,12,0.0,0.0,0.0,5.0,0.0,0.0,551.0,15.0,0.0,16.0,6.0,205.0,0.0,92.0,145.0,318.0,0.0,163.0,0.0,0.0,1516.0,348.0,582.0,586.0,0.229551,0.383905


## AADT Data

In [14]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [15]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
aadt_df.sample(10)

33 16793


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,competitors_within_1_mi,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,Rmp,dist_to_read_m
93,102141,7-ELEVEN #39804A,797 PENINSULA BLVD,HEMPSTEAD,11550,chain,CONV. STORE,-73.63171,40.69248,360594070003,36059407000,25,2321.0,56893.0,PENINSULA BLVD,14,11.0,,30.579409
12362,114448,KELLIS TOO,2601 MAIN ST,MAINE,13802,independent,RESTAURANT-HOTEL,-76.06044,42.19494,360070102002,36007010200,2,50095.0,5616.0,,16,2.0,,0.777893
15681,105754,MURPHY'S CARD & CANDY CORP,45-11 28TH AVE,ASTORIA,11103,independent,DELI-GROCERY,-73.90925,40.76338,360810147002,36081014700,117,8648.0,1947.0,48TH ST,17,1.0,,191.169991
7397,50375,KWIK FILL #084,1617 W 3RD ST,JAMESTOWN,14701,chain,CONV. STORE,-79.26403,42.09343,360130304002,36013030400,4,31012.0,10692.0,W THIRD ST,16,9.0,,0.892212
9697,59730,TOPS MARKETS #236,6150 SOUTH PARK AVE,HAMBURG,14075,chain,SUPERMARKETS,-78.82348,42.72716,360290132021,36029013202,14,33636.0,8364.0,CLARK ST,16,2.0,,69.545602
6429,115694,CORK & BARREL OF BROOKLYN INC,2813 CROPSEY AVE,BROOKLYN,11214,independent,LIQUOR,-73.98805,40.58615,360470314016,36047031401,71,1274.0,11929.0,CROPSEY AVE,14,2.0,,2.432469
6257,113376,QUICKLEE'S #30,575 N WINTON RD,ROCHESTER,14610,chain,CONV. STORE,-77.54915,43.15670,360550076002,36055007600,16,27990.0,12151.0,WINTON RD N,16,4.0,,0.037024
2666,109476,FASTRAC EG #255,1064 SCOTTSVILLE RD,ROCHESTER,14624,chain,CONV. STORE,-77.64849,43.12168,360550146022,36055014602,7,26531.0,19168.0,,14,4.0,,23.114751
4347,114205,SUNOCO,1724 WEBSTER AVE,BRONX,10457,independent,CONV. STORE,-73.90299,40.84404,360050165001,36005016500,217,4114.0,15208.0,WEBSTER AVE,14,2.0,,5.597784
13596,97684,DECAYAGUA DELI GROCERY CORP,290 ARLINGTON AVENUE,BROOKLYN,11208,independent,GROCERY STORE,-73.88381,40.68095,360471168002,36047116800,79,998.0,4390.0,FULTON ST,16,1.0,,75.345631


## EPA Data

In [16]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [17]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
80,79035,MARIA PABLITO GROCERY CORP,1551 SHERIDAN AVE,BRONX,10457,independent,GROCERY STORE,-73.91108,40.84201,"b'\x01\x01\x00\x00\x00\x8a\x02}""OzR\xc0\xd6s\xd2\xfb\xc6kD@'",360050225006,36005022500,201,156207,36,5,22500,6,106.717524,372.708396,3.503722,1.021919,0.291977,0.000000,0.000000,0.000000,0.000000,2.189826,0.0,43.733861,13.666667,36,005,022500,6
48,108836,SHUNFA DISCOUNT INC,5309 CHURCH AVE,BROOKLYN,11203,independent,GROCERY STORE,-73.92711,40.65235,b'\x01\x01\x00\x00\x00\xdar.\xc5U{R\xc0\xd6\xc5m4\x80SD@',360470866003,36047086600,123,151978,36,47,86600,3,42.375351,80.414619,3.547704,2.168041,0.000000,0.000000,0.098547,0.000000,0.985473,0.295642,0.0,35.264900,14.166667,36,047,086600,3
70,104467,OASIS DELI & GRILL CORP PLUS,814 W 187TH ST STORE 4,NEW YORK,10033,independent,DELI-GROCERY,-73.93755,40.85535,b'\x01\x01\x00\x00\x00Y\x17\xb7\xd1\x00|R\xc0\x1aQ\xda\x1b|mD@',360610273001,36061027300,84,155044,36,61,27300,1,73.027333,118.432551,26.237365,5.757644,1.384750,1.020342,4.154249,3.352552,10.349183,0.218645,0.0,22.028610,15.166667,36,061,027300,1
57,111185,7-ELEVEN #34439K,59-01 ROOSEVELT AVE,WOODSIDE,11377,chain,CONV. STORE,-73.90521,40.74541,b'\x01\x01\x00\x00\x00\xc7\x80\xec\xf5\xeeyR\xc0A\x0eJ\x98i_D@',360810259002,36081025900,142,148837,36,81,25900,2,28.056812,80.037065,57.240906,7.577844,0.501014,20.729475,4.697011,3.632355,0.250507,19.852699,0.0,25.424383,17.333333,36,081,025900,2
92,92204,AA SUPER DELI CORP,124-32 ROCKAWAY BLVD,SOUTH OZONE PARK,11420,independent,DELI-GROCERY,-73.81275,40.67535,b'\x01\x01\x00\x00\x00\xbct\x93\x18\x04tR\xc0C\xadi\xdeqVD@',360810840001,36081084000,50,156957,36,81,84000,1,15.866547,47.863204,3.953458,1.212394,0.052713,0.158138,0.184495,0.606197,1.634096,0.105426,0.0,30.422333,16.166667,36,081,084000,1


## ACS Data

In [18]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [19]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
99,101053,ALITON'S PHARMACY HOME HEALTH,10-12 SUSSEX ST,PORT JERVIS,12771,0,DRUG STORE,-74.69290,41.37437,b'\x01\x01\x00\x00\x00\x7f\xd9=yX\xacR\xc0@M-[\xeb\xafD@',360710022001,36071002200,NaN,246.0,447.0,0.111857,0.682327,0.205817,0.366337,0.031863,0.063725,0.472036,0.322275,0.150407,0.849593,0.512195,0.028455,0.459350,0.337398
88,78437,CITY MARKET & DELI INC,1245 EAST DELAVAN AVE,BUFFALO,14215,0,CONV. STORE,-78.81093,42.92269,"b'\x01\x01\x00\x00\x00\x19V\xf1F\xe6\xb3S\xc0R,\xb7\xb4\x1avE@'",360290041003,36029004100,NaN,265.0,432.0,0.108796,0.576389,0.314815,0.340062,0.255000,0.102500,0.408297,0.000000,0.441509,0.558491,0.811321,0.000000,0.188679,0.509434
15,93307,SPEEDWAY #07827,810 METROPOLITAN AVE,BROOKLYN,11211,46,CONV. STORE,-73.94166,40.71472,b'\x01\x01\x00\x00\x00\xe3\xfcM(D|R\xc0\x08\xe6\xe8\xf1{[D@',360470477002,36047047700,198884.0,937.0,1761.0,0.164111,0.822828,0.013061,0.039409,0.579857,0.274136,0.942646,0.016867,0.160085,0.839915,0.410886,0.275347,0.313767,0.558164
49,104421,NOVA DELI & GRILL CORP,145-04 LIBERTY AVE,JAMAICA,11435,0,MISCELLANEOUS,-73.80784,40.69345,b'\x01\x01\x00\x00\x00\xa4\x19\x8b\xa6\xb3sR\xc0\xa2\xb47\xf8\xc2XD@',360810206001,36081020600,56667.0,462.0,1220.0,0.179508,0.672951,0.147541,0.140063,0.052093,0.024186,0.711838,0.123632,0.354978,0.645022,0.854978,0.062771,0.082251,0.424242
26,58010,THE WINE SELLAR,3843 UNION RD SUITE 100,BUFFALO,14225,0,LIQUOR,-78.75317,42.91902,b'\x01\x01\x00\x00\x00\x03\x95\xf1\xef3\xb0S\xc0Z/\x86r\xa2uE@',360290100031,36029010003,75909.0,824.0,1434.0,0.169456,0.625523,0.205021,0.083792,0.182642,0.117736,0.695918,0.040078,0.739078,0.260922,0.588592,0.084951,0.326456,0.013350


## Master Data Sample

In [20]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
master_df.sample(5)

(14341, 58)
Index(['agtno', 'avg_daily_sales', 'chain', 'bustype', 'longitude', 'latitude',
       'bg_geoid', 'tract_geoid', 'region', 'competitors_within_1_mi',
       'aadt_count', 'aadt_functional_class', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion',
       'age_30_64_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns

,agtno,avg_daily_sales,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,region,competitors_within_1_mi,aadt_count,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,total_population,age_18_29_proportion,age_30_64_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,family_households_proportion,no_vehicle_households_proportion,D1B,D1C,D3A,NatWalkInd,commercial_places_within_1_mi,community_places_within_1_mi,rac_c000,rac_ce01_prop,rac_ce02_prop,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01_prop,wac_ce02_prop
10653,67032,3244.797260,independent,CONV. STORE,-73.31986,40.73677,361031228022,36103122802,Long Island,23.0,743.0,19,2.0,12.291559,134865.0,2294.0,0.187010,0.634263,0.032470,0.204511,0.085714,0.648021,0.011952,0.988789,0.836323,0.000000,12.639902,2.156389,26.059785,10.500000,141.0,19.0,1213.0,0.099753,0.246496,4.0,0.0,0.0,52.0,0.0,0.0,118.0,0.0,0.0,2.0,0.0,15.0,0.0,53.0,12.0,36.0,19.0,193.0,22.0,0.0,526.0,0.363118,0.397338
7757,18182,6569.224658,independent,STATIONERY-CARD,-73.88515,40.96058,361190008014,36119000801,Mid-Hudson,22.0,2447.0,17,2.0,1.049937,155598.0,664.0,0.027108,0.658133,0.024079,0.123494,0.278614,0.674419,0.000000,0.622318,0.802575,0.072961,26.630607,2.760490,30.033853,17.166667,101.0,41.0,326.0,0.110429,0.199387,0.0,0.0,0.0,4.0,0.0,0.0,35.0,1.0,0.0,12.0,5.0,0.0,1.0,0.0,0.0,0.0,0.0,11.0,11.0,0.0,80.0,0.287500,0.450000
10587,66355,2958.921918,independent,GAS,-73.39823,40.73248,361031224083,36103122408,Long Island,2.0,18898.0,16,4.0,0.922114,NaN,51.0,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.509804,0.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,27.0,11.0,29.0,0.137931,0.344828,0.0,0.0,17.0,1048.0,637.0,478.0,80.0,340.0,18.0,0.0,124.0,233.0,20.0,927.0,39.0,23.0,43.0,3.0,108.0,0.0,4138.0,0.072982,0.162397
12733,81627,559.501370,chain,SUPERMARKETS,-73.92324,40.71434,360810535021,36081053502,New York City,62.0,12671.0,14,2.0,41.637351,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,778.0,103.0,130.0,0.184615,0.338462,0.0,0.0,0.0,40.0,287.0,591.0,106.0,270.0,0.0,0.0,37.0,122.0,88.0,25.0,0.0,0.0,165.0,7.0,4.0,47.0,1789.0,0.158189,0.290665
10870,69153,2091.020548,chain,SUPERMARKETS,-75.45462,44.34086,360894927001,36089492700,North Country,3.0,9987.0,04,6.0,74.319410,88964.0,565.0,0.145133,0.362832,0.022013,0.145873,0.094050,0.355752,0.233831,1.000000,0.829876,0.132780,0.156292,0.056700,2.043562,6.666667,37.0,16.0,227.0,0.110132,0.286344,0.0,0.0,0.0,0.0,0.0,0.0,222.0,46.0,0.0,0.0,0.0,1.0,0.0,0.0,108.0,3.0,0.0,33.0,4.0,0.0,417.0,0.263789,0.400480


## Missingness and Variance Check

In [21]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [22]:
numeric_features = ['competitors_within_1_mi', 'aadt_count', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion', 'age_30_64_proportion',
       'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
       'labor_force_participation_rate', 'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns15', 'wac_cns16', 'wac_cns17', 'wac_cns18', 'wac_cns19',
       'wac_cns20', 'wac_c000', 'wac_ce01_prop', 'wac_ce02_prop']

categorical_features = ['chain', 'bustype', 'aadt_functional_class']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

,pct_missing,pct_zero,pct_mode,mode,mean,std,iqr,cv,n_unique
NatWalkInd,18.290217,0.000000,2.227343,16.000000,12.550549,4.124931,6.666667,0.328665,114
D3A,18.290217,0.000000,0.128008,27.524029,21.927281,12.297690,17.052359,0.560840,6207
D1C,18.290217,0.146433,0.179211,0.000000,28.550338,136.314702,10.901196,4.774539,6192
D1B,18.290217,0.216163,0.264550,0.000000,40.631699,63.115681,53.089343,1.553361,6186
median_household_income,10.424657,0.000000,1.370076,250001.000000,92799.536899,47438.643700,58289.750000,0.511195,5833
family_households_proportion,1.317900,0.160379,0.847937,1.000000,0.612610,0.194447,0.265905,0.317408,6761
no_vehicle_households_proportion,1.317900,11.937801,12.097230,0.000000,0.261447,0.280911,0.399796,1.074447,5922
poverty_rate,1.317900,5.892197,5.970888,0.000000,0.143138,0.140365,0.156161,0.980628,6685
owner_occupied_proportion,1.317900,5.348302,5.419729,0.000000,0.536270,0.317095,0.570462,0.591298,6339
unemployment_rate,1.241197,17.662646,17.884629,0.000000,0.065368,0.074246,0.075611,1.135812,5357


In [23]:
categorical_summary['chain']

{'pct_missing': np.float64(0.0),
 'distribution':                    pct
 chain                 
 independent  73.579248
 chain        26.420752}

## Correlation

In [24]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,owner_occupied_proportion,renter_occupied_proportion,-1.000,1.000
1,wac_c000,wac_ce03,0.974,0.974
2,arts_and_entertainment_within_1_mi,shopping_within_1_mi,0.971,0.971
3,shopping_within_1_mi,restaurant_within_1_mi,0.970,0.970
4,shopping_within_1_mi,social_or_community_service_within_1_mi,0.968,0.968
5,restaurant_within_0.5_mi,restaurant_within_1_mi,0.965,0.965
6,arts_and_entertainment_within_1_mi,social_or_community_service_within_1_mi,0.962,0.962
7,arts_and_entertainment_within_1_mi,restaurant_within_1_mi,0.955,0.955
8,restaurant_within_1_mi,bar_within_1_mi,0.953,0.953
9,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,0.951,0.951
